In [1]:
import numpy as np
import pandas as pd

In [4]:
import pandas as pd

def read_excel(file_path):
    try:
        # Use ExcelFile to access metadata like sheet names
        with open(file_path) as xls:
            # print(f"Available sheets: {xls.sheet_names}")
            
            # Read the specific sheet from the ExcelFile object
            df = pd.read_csv(xls) 
        return df
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return None


In [11]:
df = read_excel("../Data/email.csv")
df.head()

,HR Name,Email,Company,Hiring Role,Last Email Sent Date
0,saisrushik,saisrushik98@gmail.com,ABC,AI Engineer,NaN
1,saisrushik,saisrushik2003@gmail.com,XYZ,Python Developer,NaN


In [12]:
def get_hr_names(df):
    """
    Extracts the HR team name from the DataFrame.
    
    Parameters:
    df (pd.DataFrame): The DataFrame containing the data.
    
    Returns:
    str: The HR team name if found, otherwise None.
    """
    try:
        hr_team_name = df['HR Name']
        return hr_team_name
    except Exception as e:
        print(f"Error extracting HR team name: {e}")
        return None

In [14]:
get_hr_names(df)

0    saisrushik
1    saisrushik
Name: HR Name, dtype: str

pdf REader

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader
import pymupdf as fitz
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")

def read_resume(file_path):
        try:
            # --- LangChain loader: returns one Document per page ---
            loader = PyMuPDFLoader(file_path)
            resume_data = loader.load()

            # --- PyMuPDF: extract links per page ---
            pdf_doc = fitz.open(file_path)

            for page_index, page in enumerate(pdf_doc):
                links = page.get_links()  # list of dicts with 'uri' key for hyperlinks
                urls = [link["uri"] for link in links if link.get("uri")]

                if urls and page_index < len(resume_data):
                    hyperlink_section = (
                        "\n\nHyperlinks on this page:\n"
                        + "\n".join(f"- {url}" for url in urls)
                    )
                    resume_data[page_index].page_content += hyperlink_section
                    resume_data[page_index].metadata["hyperlinks"] = urls

            pdf_doc.close()
            return resume_data

        except FileNotFoundError:
            print(f"Error: The file at {file_path} was not found.")
            return None
        except Exception as e:
            print(f"Error reading resume file: {e}")
            return None

In [5]:
docs = read_resume("../Data/Resume - ML.pdf")
print(docs)

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-23T22:17:18+05:30', 'source': '../Data/Resume - ML.pdf', 'file_path': '../Data/Resume - ML.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': 'GOVINDGARI SAI SRUSHIK 21BCE8659', 'subject': '', 'keywords': '', 'moddate': '2026-03-23T22:17:18+05:30', 'trapped': '', 'modDate': "D:20260323221718+05'30'", 'creationDate': "D:20260323221718+05'30'", 'page': 0, 'hyperlinks': ['https://sites.google.com/view/sai-srushik', 'mailto:+91%2094916377285', 'mailto:saisrushik98@gmail.com', 'https://www.linkedin.com/in/sai-srushik/', 'https://github.com/saisrushik', 'https://sites.google.com/view/sai-srushik', 'https://scholar.google.com/citations?user=mLj1zRQAAAAJ&hl=en', 'https://github.com/saisrushik/QA-ChatBot-using-Groq-OpenAI-Ollama', 'https://github.com/saisrushik/youtube-video-summarizer/', 'https://github.com/saisrushik/Pneumonia-Diesease-Prediction', 'https://link

In [11]:
docs[0].page_content

"Govindgari Sai Srushik \n+91 9491637728 | saisrushik98@gmail.com | LinkedIn | GitHub | Portfolio | Google Scholar \nHighly motivated AI/ML Engineer with deep expertise in Generative AI research and development. Proven ability to design \nand implement complex AI Systems. Actively leveraging Python, TensorFlow, and Langchain to drive projects from research \n(publication achieved) to deployment, and actively upskilling in MLOps & Azure Cloud Services. \nEducation \nVellore Institute of Technology (VIT-AP), India  \n \n \n \n \n \n(CGPA: 8.73/10) \nB. Tech CSE with Specialization in AI & ML.  \nPine Grove College, India  \n \n \n \n \n \n \n \n \n(Percentage: 97%) \nHigher Secondary \n \nWork Experience \n➢ Programmer Analyst Trainee – Cognizant Inc. (August 2025 – Present) \no Contributed to high-stakes testing to verify functionality of the application, ensuring quality standards for \ncritical software releases. \no Architected an End-to-End Generative AI Pipeline to generate Playwri

In [9]:
docs[0].metadata["hyperlinks"]

['https://sites.google.com/view/sai-srushik',
 'mailto:+91%2094916377285',
 'mailto:saisrushik98@gmail.com',
 'https://www.linkedin.com/in/sai-srushik/',
 'https://github.com/saisrushik',
 'https://sites.google.com/view/sai-srushik',
 'https://scholar.google.com/citations?user=mLj1zRQAAAAJ&hl=en',
 'https://github.com/saisrushik/QA-ChatBot-using-Groq-OpenAI-Ollama',
 'https://github.com/saisrushik/youtube-video-summarizer/',
 'https://github.com/saisrushik/Pneumonia-Diesease-Prediction',
 'https://link.springer.com/chapter/10.1007/978-3-031-78922-9_29',
 'https://learn.microsoft.com/api/credentials/share/en-us/saisrushik/5B1F2867F3F3FC03?sharingId=8FBA3F38B569774C']

In [ ]:
web_links = [link for link in docs[0].metadata['hyperlinks'] if link.startswith("http")]
mail_links = [l for l in hyperlinks if l.startswith("mailto:")]

def extract_email(mailto_links):
    for link in mailto_links:
        decoded = unquote(link)                      # decode %40 -> @, etc.
        match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', decoded)
        if match:
            return match.group()
    return None

def extract_phone(mailto_links):
    for link in mailto_links:
        decoded = unquote(link)                      # decode %20 -> space, etc.
        match = re.search(r'[\+\d][\d\s\-]{8,14}\d', decoded)
        if match:
            # Strip spaces for clean output
            return re.sub(r'\s+', '', match.group())
    return None

# Or as a dict for easier lookup
import re
links_dict = {
    "email":    extract_email(mail_links),
    "phone":    extract_phone(mail_links),
    "linkedin": next((l for l in web_links if "linkedin.com" in l), None),
    "github":   next((l for l in web_links if "github.com" in l), None),
    "portfolio": next((l for l in web_links if "sites.google.com" in l), None),
    "scholar":  next((l for l in web_links if "scholar.google.com" in l), None),
}

print(links_dict)


{'linkedin': 'https://www.linkedin.com/in/sai-srushik/', 'github': 'https://github.com/saisrushik', 'portfolio': 'https://sites.google.com/view/sai-srushik', 'scholar': 'https://scholar.google.com/citations?user=mLj1zRQAAAAJ&hl=en'}
